# Task 3: Multi-Agent Financial Research System
**CDAZZDEV Senior MLE Assessment · Agentic Workflows** · framework: **LangGraph** · LLMs: Groq free tier (`gpt-oss-120b` / `gpt-oss-20b`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/CDAZZDEV-MLE-YourName/blob/main/task3_agentic/notebooks/Task3_Multi_Agent_Research.ipynb)

| Part | What this notebook shows | Code |
|---|---|---|
| 3A | Five traced tools; one autonomous ReAct agent; a visible observe→replan cycle (including recovery from a tool failure); a three-section report | `tools.py`, `agent.py`, `research.py` |
| 3B | Agent A (Data Analyst) and Agent B (Research Writer) with tool allowlists enforced at execution time; Pydantic handoffs; a critique loop | `multi_agent.py`, `schemas.py` |
| 3C | Short-term memory (a follow-up answered from context with 0 tool calls); a persistent JSON cache keyed by ticker and date; `agent_trace.jsonl` | `memory.py`, `tracing.py` |
| Bonus | A Streamlit dashboard that reads the trace | `dashboard/app.py` |

**Design choice:** the ReAct loop is an explicit LangGraph `StateGraph` (`agent ⇄ tools`), not a prebuilt helper. This lets the tool executor deny calls outside an agent's role, apply a step budget, and turn every tool error into an observation the agent can react to.

In [1]:
import os, sys, subprocess, json, time
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/YOUR_GITHUB_USERNAME/CDAZZDEV-MLE-YourName", "/content/repo"], check=True)
    os.chdir("/content/repo/task3_agentic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env", override=True)
sys.path.insert(0, str(Path.cwd()))
print("cwd:", Path.cwd(), "| GROQ_API_KEY", "set" if os.getenv("GROQ_API_KEY") else "MISSING")

cwd: c:\Users\Techon Computers\Documents\Assesment\task3_agentic | GROQ_API_KEY set


In [2]:
import logging
import pandas as pd
from IPython.display import Markdown, display
from research_agents import config, tools as T
from research_agents.tracing import TRACER
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 120)

# Fresh trace for this notebook run (the previous one is archived, not deleted).
if config.TRACE_PATH.exists():
    config.TRACE_PATH.rename(config.TRACE_PATH.with_name(f"agent_trace_prev_{int(time.time())}.jsonl"))
TICKER = "MSFT"
print("models:", config.MODELS)

models: {'research_agent': 'openai/gpt-oss-120b', 'data_analyst': 'openai/gpt-oss-20b', 'research_writer': 'openai/gpt-oss-120b', 'sentiment_tool': 'openai/gpt-oss-20b'}


---
# 3A: Tool-using research agent
## 3A.1 The five tools: each is callable and returns a typed result
Every tool is wrapped by `@traced`, which handles timing, JSONL logging, fault injection, and a last-resort exception guard. A failure comes back as `{"error", "hint"}` or an empty list and never as an exception, so the agent can *observe* the failure and re-plan.

In [3]:
TRACER.set_agent("tool_smoke_test")
checks = {
    "get_price_data": T.get_price_data(ticker=TICKER, period="1y"),
    "get_news": T.get_news(ticker=TICKER, n=5),
    "calculate_volatility": T.calculate_volatility(ticker=TICKER, window=30),
    "web_search": T.web_search(query=f"{TICKER} analyst outlook risks", max_results=3),
}
checks["llm_sentiment"] = T.llm_sentiment(headlines=[n["headline"] for n in checks["get_news"]])
pd.DataFrame([{"tool": k, "return type": type(v).__name__,
               "items/keys": len(v), "preview": json.dumps(v, default=str)[:150]} for k, v in checks.items()])

    ✓ [tool_smoke_test] TOOL get_price_data({"ticker": "MSFT", "period": "1y"}) -> ok in 2292 ms | {"ticker": "MSFT", "period": "1y", "n_bars": 252, "start": "2025-10-07", "end": "2026-10-07", "last_close": 5…
    ✓ [tool_smoke_test] TOOL get_news({"ticker": "MSFT", "n": 5}) -> ok in 6241 ms | [{"headline": "Microsoft Xbox broadens entertainment push with new XP division (MSFT:NASDAQ)", "publisher": "…
    ✓ [tool_smoke_test] TOOL calculate_volatility({"ticker": "MSFT", "window": 30}) -> ok in 550 ms | {"ticker": "MSFT", "window_days": 30, "annualised_vol_pct": 22.01, "one_year_vol_pct": 32.3, "vol_percentile_…
    ✓ [tool_smoke_test] TOOL web_search({"query": "MSFT analyst outlook risks", "max_results": 3}) -> ok in 5108 ms | [{"title": "Microsoft (MSFT) Stock Forecast & Analyst Price Targets", "snippet": "Stock forecasts and analyst…
    ✓ [tool_smoke_test] TOOL llm_sentiment({"headlines": ["Microsoft Xbox broadens entertainment push with new XP division (MSFT:NAS…) -> ok in 5135 ms 

,tool,return type,items/keys,preview
0,get_price_data,dict,12,"{""ticker"": ""MSFT"", ""period"": ""1y"", ""n_bars"": 252, ""start"": ""2025-10-07"", ""end"": ""2026-10-07"", ""last_close"": 529.76, ..."
1,get_news,list,5,"[{""headline"": ""Microsoft Xbox broadens entertainment push with new XP division (MSFT:NASDAQ)"", ""publisher"": ""Seeking..."
2,calculate_volatility,dict,7,"{""ticker"": ""MSFT"", ""window_days"": 30, ""annualised_vol_pct"": 22.01, ""one_year_vol_pct"": 32.3, ""vol_percentile_1y"": 32..."
3,web_search,list,3,"[{""title"": ""Microsoft (MSFT) Stock Forecast & Analyst Price Targets"", ""snippet"": ""Stock forecasts and analyst price ..."
4,llm_sentiment,dict,6,"{""score"": 0.659, ""label"": ""positive"", ""n_headlines"": 5, ""counts"": {""positive"": 4, ""neutral"": 0, ""negative"": 1}, ""mos..."


In [4]:
print("Failure paths return structured errors instead of raising:")
print(" bad ticker  ->", T.get_price_data(ticker="NOTATICKERZZ", period="1y"))
print(" bad window  ->", T.calculate_volatility(ticker=TICKER, window=2))
print(" no headlines->", T.llm_sentiment(headlines=[]))

Failure paths return structured errors instead of raising:


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKERZZ"}}}
$NOTATICKERZZ: No data found, symbol may be delisted


    ✗ [tool_smoke_test] TOOL get_price_data({"ticker": "NOTATICKERZZ", "period": "1y"}) -> error in 1461 ms | {"error": "LookupError: no price data for 'NOTATICKERZZ' (period 2y)", "hint": "check the arguments or use an…
 bad ticker  -> {'error': "LookupError: no price data for 'NOTATICKERZZ' (period 2y)", 'hint': 'check the arguments or use an alternative tool'}
    ✗ [tool_smoke_test] TOOL calculate_volatility({"ticker": "MSFT", "window": 2}) -> error in 0 ms | {"error": "window 2 out of range", "hint": "use 5-252 trading days"}
 bad window  -> {'error': 'window 2 out of range', 'hint': 'use 5-252 trading days'}
    ✗ [tool_smoke_test] TOOL llm_sentiment({"headlines": []}) -> error in 0 ms | {"error": "no headlines supplied", "hint": "fetch headlines with get_news or web_search first"}
 no headlines-> {'error': 'no headlines supplied', 'hint': 'fetch headlines with get_news or web_search first'}


## 3A.2 Autonomous run with an injected tool failure
The agent gets the brief's query and all five tools; **no tool order is coded anywhere**. To show the "tool fails → try an alternative" requirement on a real run, we **inject one simulated outage of `get_news`** (`set_fault_injection`, clearly labelled in the trace).

How to read the trace: `THINK` is the model's own reasoning summary returned by the provider, `ACT` is the tool call it chose, and `✓/✗` is the observation. The observe→replan cycle is the `✗ get_news` line followed by a THINK that chooses an alternative.

In [5]:
from research_agents.research import run_single_agent
from research_agents.render import report_markdown
T.set_fault_injection({"get_news": 1})
TRACER.new_session()
session = run_single_agent(TICKER)
T.set_fault_injection({})

=== Research agent | thread single-02057e ===
  [research_agent] step 1 THINK (openai/gpt-oss-120b): We need price data, news, sentiment, volatility, web search.
  [research_agent] step 1 ACT  : get_price_data({"period": "3mo", "ticker": "MSFT"})
    ✓ [research_agent] TOOL get_price_data({"ticker": "MSFT", "period": "3mo"}) -> ok in 398 ms | {"ticker": "MSFT", "period": "3mo", "n_bars": 66, "start": "2026-07-07", "end": "2026-10-07", "last_close": 5…
  [research_agent] step 2 THINK (openai/gpt-oss-120b): Need news, sentiment, volatility, web search.
  [research_agent] step 2 ACT  : get_news({"n": 10, "ticker": "MSFT"})
    ✗ [research_agent] TOOL get_news({"ticker": "MSFT", "n": 10}) -> error in 0 ms | {"error": "get_news unavailable (simulated provider outage)", "hint": "use an alternative tool or retry with …
  [research_agent] step 3 THINK (openai/gpt-oss-120b): Need news via web_search.
  [research_agent] step 3 ACT  : web_search({"max_results": 5, "query": "Microsoft Corp recent 

In [6]:
# Extract the observe -> replan evidence from the run's messages
from langchain_core.messages import AIMessage, ToolMessage
seq = []
for m in session.messages:
    if isinstance(m, AIMessage) and m.tool_calls:
        seq.append(("DECIDE", ", ".join(c["name"] for c in m.tool_calls), m.additional_kwargs.get("reasoning", "")[:140]))
    elif isinstance(m, ToolMessage):
        status = "ERROR" if '"error"' in str(m.content)[:40] else "ok"
        seq.append(("OBSERVE", m.name, f"{status}: {str(m.content)[:110]}"))
display(pd.DataFrame(seq, columns=["phase", "tool", "detail"]))
order = [c["name"] for m in session.messages if isinstance(m, AIMessage) for c in (m.tool_calls or [])]
print("Tool order chosen by the agent:", " -> ".join(order))

,phase,tool,detail
0,DECIDE,get_price_data,"We need price data, news, sentiment, volatility, web search."
1,OBSERVE,get_price_data,"ok: {""ticker"": ""MSFT"", ""period"": ""3mo"", ""n_bars"": 66, ""start"": ""2026-07-07"", ""end"": ""2026-10-07"", ""last_close"": 52"
2,DECIDE,get_news,"Need news, sentiment, volatility, web search."
3,OBSERVE,get_news,"ERROR: {""error"": ""get_news unavailable (simulated provider outage)"", ""hint"": ""use an alternative tool or retry with d"
4,DECIDE,web_search,Need news via web_search.
5,OBSERVE,web_search,"ok: [{""title"": ""Windows and Surface October 2026 news - news.microsoft.com"", ""snippet"": ""Join us for the latest on"
6,DECIDE,calculate_volatility,"We have price data, news via web search, need volatility."
7,OBSERVE,calculate_volatility,"ok: {""ticker"": ""MSFT"", ""window_days"": 30, ""annualised_vol_pct"": 22.01, ""one_year_vol_pct"": 32.3, ""vol_percentile_1"
8,DECIDE,llm_sentiment,"We have price data, volatility, web search results (no news sentiment). Need sentiment: we can use headlines from we..."
9,OBSERVE,llm_sentiment,"ok: {""score"": 0.0, ""label"": ""neutral"", ""n_headlines"": 5, ""counts"": {""positive"": 0, ""neutral"": 5, ""negative"": 0}, """


Tool order chosen by the agent: get_price_data -> get_news -> web_search -> calculate_volatility -> llm_sentiment -> web_search


In [7]:
assert session.report is not None, "report extraction failed"
display(Markdown(report_markdown(session.report)))

## Research report — MSFT
### 1. Financial Health Summary
MSFT closed at $529.76, up 36.5% over the past three months and trading 6.84% above its 50‑day SMA and 22.56% above its 200‑day SMA. RSI is 68.21 (near‑overbought) and MACD remains positive (+1.64). The 30‑day annualised volatility is 22.01%, well below the 1‑year level of 32.3%, and beta versus SPY is 0.95. Max drawdown in the period was modest at -5.15%, indicating limited downside so far.

**Market sentiment.** LLM sentiment on recent headlines is neutral (score 0.0). Analyst commentary from web searches highlights accelerating Azure and AI demand, but flags execution risk around AI‑capex and growing regulatory scrutiny, creating a cautious outlook despite strong momentum.

### 2. Top Three Risks (90 days)
**1. AI‑capex execution risk** — likelihood *high*

Heavy investment in AI infrastructure may not translate into the expected revenue growth, potentially missing the 39‑40% cloud growth guidance and pressuring margins.

- [web_search] "Azure & AI demand still accelerating (Futurum, Jul 31 2026)"
- [web_search] "AI‑capex risk highlighted in Q4 FY‑26 earnings call (MSDynamicsWorld, Aug 6 2026)"

**2. Regulatory pressure on AI and data** — likelihood *medium*

Increasing global regulation (EU AI Act, US antitrust probes) could force product changes, add compliance costs, or result in fines, impacting Microsoft’s AI rollout and profitability.

- [web_search] "AI regulation cited as risk factor for FY‑27 guidance (RiskIntelligence, Jun 2 2026)"

**3. Market‑wide volatility rebound** — likelihood *medium*

Current low volatility (22.0% annualised, 32.9th percentile) makes the stock vulnerable to a sudden spike in market volatility, which could trigger a sell‑off in high‑beta tech stocks.

- [calculate_volatility] "30‑day annualised vol 22.01% vs 1‑yr vol 32.3% (vol percentile 32.9)"
- [calculate_volatility] "beta vs SPY 0.95"

### 3. Hedge Strategy Recommendation
**protective collar** — Buy a 90‑day $480 put and sell a 90‑day $560 call on MSFT, both expiring in October 2026, creating a floor at ~‑9% and a ceiling at ~+5.5% of current price.

*Data rationale:* Low 30‑day volatility (22.0%) reduces option premiums; beta 0.95 means market moves affect MSFT; 1‑day 95% VaR of 2.84% indicates a modest downside risk that the $480 put caps. The collar net cost is near zero after the call premium offsets the put cost.

*Cost / trade-off:* Net premium ≈ $0.5 per share (≈0.1% of price); downside limited to -9%, upside capped at +5.5%.

*Sources: get_price_data, calculate_volatility, llm_sentiment, web_search*

> Educational output generated by an LLM agent system — not investment advice.

---
# 3B: Multi-agent coordination
| Agent | Tools (enforced) | Output |
|---|---|---|
| A · Data Analyst | `get_price_data`, `calculate_volatility`, `llm_sentiment` | `DataBrief` (Pydantic) |
| B · Research Writer | `web_search`, `get_news` | `FinalReport` (Pydantic) |

**Why neither agent can do it alone:** A has no news source, so it can't score sentiment until B supplies headlines. B has no price tools, so every number in its report must come from A. The **critique loop** connects them: B sends a `ClarificationRequest` (with the headlines B collected), A answers with a `ClarificationResponse` computed with A's tools, and B incorporates it.

**Handoff integrity:** the numeric fields of `DataBrief` and `ClarificationResponse` are copied from tool JSON (`snapshots_from_tools`). The LLM writes only the narrative fields, so no figure passed between agents is re-typed by a model.

## 3B.1 Tool restriction is enforced at execution time, not only at binding time

In [8]:
from langchain_core.messages import AIMessage as _AI
from research_agents.agent import ToolExecutor
TRACER.set_agent("data_analyst")
forged = _AI(content="", tool_calls=[{"name": "web_search", "args": {"query": "MSFT"}, "id": "forged-1"}])
denied = ToolExecutor("data_analyst")({"messages": [forged], "steps": 1})["messages"][0]
print("Agent A attempting web_search ->", denied.content)
TRACER.set_agent("research_writer")
forged = _AI(content="", tool_calls=[{"name": "get_price_data", "args": {"ticker": "MSFT"}, "id": "forged-2"}])
print("Agent B attempting get_price_data ->", ToolExecutor("research_writer")({"messages": [forged], "steps": 1})["messages"][0].content)

    ⛔ [data_analyst] TOOL web_search({"query": "MSFT"}) -> denied in 0 ms | {"error": "PermissionError: tool 'web_search' is not available to data_analyst", "allowed_tools": ["calculate…
Agent A attempting web_search -> {"error": "PermissionError: tool 'web_search' is not available to data_analyst", "allowed_tools": ["calculate_volatility", "get_price_data", "llm_sentiment"]}
    ⛔ [research_writer] TOOL get_price_data({"ticker": "MSFT"}) -> denied in 0 ms | {"error": "PermissionError: tool 'get_price_data' is not available to research_writer", "allowed_tools": ["ge…
Agent B attempting get_price_data -> {"error": "PermissionError: tool 'get_price_data' is not available to research_writer", "allowed_tools": ["get_news", "web_search"]}


## 3B.2 End-to-end pipeline: query → final report with no manual steps (full message trace)

In [9]:
from research_agents.multi_agent import run_pipeline
t0 = time.time()
state = run_pipeline(TICKER, use_cache=False)     # force a fresh run; this run writes the cache
print(f"\nPipeline finished in {time.time() - t0:.0f} s, no manual intervention.")


════════════════════════════════════════════════════════════════════════════════════════════════════
AGENT A · Data Analyst · tools: get_price_data, calculate_volatility, llm_sentiment
════════════════════════════════════════════════════════════════════════════════════════════════════
  [data_analyst] step 1 THINK (openai/gpt-oss-20b): Need price data, volatility, VaR, beta. Use get_price_data for 1y. Then calculate_volatility with window 30 and default 1y.
  [data_analyst] step 1 ACT  : get_price_data({"period": "1y", "ticker": "MSFT"})
    ✓ [data_analyst] TOOL get_price_data({"ticker": "MSFT", "period": "1y"}) -> ok in 327 ms | {"ticker": "MSFT", "period": "1y", "n_bars": 252, "start": "2025-10-07", "end": "2026-10-07", "last_close": 5…
  [data_analyst] step 2 THINK (openai/gpt-oss-20b): Need volatility.
  [data_analyst] step 2 ACT  : calculate_volatility({"ticker": "MSFT", "window": 30})
    ✓ [data_analyst] TOOL calculate_volatility({"ticker": "MSFT", "window": 30}) -> ok in 556 

In [10]:
final = state["final_report"]
req, resp = state["clarification_request"], state["clarification_response"]
print("Critique loop executed:", req is not None and resp is not None)
print("  B asked :", req.question)
print("  A answer:", resp.answer_summary)
display(Markdown(report_markdown(final)))

Critique loop executed: True
  B asked : Can you score the sentiment of the collected Microsoft headlines to fill the missing sentiment data?
  A answer: The llm_sentiment tool returned a positive overall score of 0.538, with 4 positive, 1 neutral, and 1 negative headline. It identified the most negative headline as "Michael Burry says Microsoft uses spending metrics ‘as a lever’ to get its way (MSFT:NASDAQ)" and the most positive headlines as "Microsoft Xbox broadens entertainment push with new XP division (MSFT:NASDAQ)", "JPM sees software rally extending, favors Microsoft, ServiceNow, Snowflake (MSFT:NASDAQ)", and "These 3 Catalysts Make Microsoft Stock a Bargain Before Year-End".


## Research report — MSFT
### 1. Financial Health Summary
MSFT closed at $529.76, trading 22.56% above its 200‑day SMA and 6.84% above its 50‑day SMA, indicating strong momentum. The 30‑day annualised volatility is 22.01%, well below the 1‑year level of 32.30%, while beta versus SPY is 0.95, suggesting near‑market movement. RSI‑14 sits at 68.21 (over‑bought) and Bollinger %B is 0.986. The stock’s max drawdown over the past year was –34.5%, but price recovery is evident.

**Market sentiment.** Sentiment analysis of six recent headlines yields a positive overall score of 0.538, with four positive, one neutral and one negative headline, confirming a broadly upbeat market mood toward Microsoft despite a single critical comment.

### 2. Top Three Risks (90 days)
**1. Escalating U.S. antitrust probe of Azure, Copilot and cloud bundling** — likelihood *high*

The FTC has expanded its investigation to cover Microsoft’s cloud licensing, AI bundling and OpenAI partnership, raising the risk of fines, remedial licensing changes or forced un‑bundling that could dent Azure revenue and investor confidence within the next quarter.

- [web_search] "Microsoft Antitrust Probe Widens: 6 Rivals Queried" – FTC now examining cloud licensing, AI bundling and the OpenAI partnership (June 2026)
- [DataBrief] "MSFT’s closing price is 22.56% above its 200‑day SMA while the 30‑day annualised volatility (22.01%) is below its 1‑year level (32.30%)"

**2. Competitive pressure from AI‑chip rivals and Apple’s AI‑enhanced Macs** — likelihood *medium*

Apple’s launch of AI‑powered Macs and aggressive Nvidia chip advances threaten Microsoft’s Surface hardware narrative and could weaken the AI‑hardware growth story that underpins the current rally.

- [get_news] "MSFT Stock On 4-Day Winning Streak: Microsoft Takes On Apple's Macs With Nvidia-Powered Surface Ultra, More On-Device AI" (Seeking Alpha, 8 Oct 2026)
- [DataBrief] "The RSI‑14 is 68.21 and Bollinger %B is 0.986, indicating the stock is in over‑bought territory despite a positive MACD histogram of 1.638"

**3. Macro‑tech earnings slowdown and S&P 500 underperformance** — likelihood *medium*

A broader slowdown in tech earnings or a pull‑back in the S&P 500 could drag MSFT lower, especially given its near‑market beta of 0.95 and its current over‑bought technical condition.

- [web_search] "Microsoft Stock Is on Track to Trail the S&P 500 for a 3rd Straight Year" (The Motley Fool, 7 Oct 2026)
- [DataBrief] "beta_vs_spy":0.95

### 3. Hedge Strategy Recommendation
**protective put spread** — Buy a 90‑day ATM MSFT put at approx. $530 strike and sell a 90‑day put 5% lower at approx. $500 strike; net premium ~ $2.8 per share.

*Data rationale:* 30‑day annualised vol 22.01% → implied 90‑day vol ~23‑24%; beta 0.95 means near‑market exposure; RSI 68 indicates over‑bought, so downside protection is prudent; spread limits cost while capping loss at $500 (~5% below current price).

*Cost / trade-off:* Net premium ~0.5% of stock price; caps upside above $540 but preserves most upside while providing a floor around $500.

### Critique loop — how Agent A's answer was used
We asked Agent A to score sentiment of six Microsoft headlines; Agent A returned a positive sentiment score of 0.538 (4 positive, 1 neutral, 1 negative). This allowed us to add a market_sentiment_summary and cite the sentiment evidence, confirming overall bullish mood and strengthening the view that the primary risks are regulatory, competitive and macro rather than sentiment‑driven.

*Sources: DataBrief, ClarificationResponse, web_search, get_news*

> Educational output generated by an LLM agent system — not investment advice.

---
# 3C: Memory and observability
## 3C.1 Short-term memory
The single agent's graph uses a LangGraph `InMemorySaver` checkpointer keyed by `thread_id`, so earlier tool observations stay in context. We ask a follow-up and count tool calls made *during the follow-up*.

In [11]:
from research_agents.research import follow_up
TRACER.new_session()
n_trace_before = sum(1 for _ in config.TRACE_PATH.open(encoding="utf-8"))
answer, n_tools = follow_up(session, "Follow-up: what 30-day annualised volatility and beta did you find, "
                                     "and is current volatility above or below its 1-year level?")
n_trace_after = sum(1 for _ in config.TRACE_PATH.open(encoding="utf-8"))
display(Markdown(answer))
print(f"Tool calls during follow-up: {n_tools} | new trace lines: {n_trace_after - n_trace_before}")
assert n_tools == 0, 'follow-up should be answered from context'

  [research_agent] step 1 ANSWER (openai/gpt-oss-20b): **30‑day annualised volatility & beta for MSFT** | Metric | Value | 1‑year level | Above / Below 1‑year? | |--------|-------|--------------|------------------------| | **30‑day annualised volatility** | **22.01 %** | 32.3 % | **Below** | | **Beta vs. SPY** | 0.95 | – | – | The 30‑day volatility (22…


**30‑day annualised volatility & beta for MSFT**

| Metric | Value | 1‑year level | Above / Below 1‑year? |
|--------|-------|--------------|------------------------|
| **30‑day annualised volatility** | **22.01 %** | 32.3 % | **Below** |
| **Beta vs. SPY** | 0.95 | – | – |

The 30‑day volatility (22 %) is lower than the 1‑year historical volatility (32.3 %), indicating that MSFT is currently in a relatively low‑volatility regime.

Tool calls during follow-up: 0 | new trace lines: 0


## 3C.2 Persistent memory: the second run loads the cached brief instead of re-running tools

In [12]:
from research_agents import memory
print("Cached file:", memory.cache_path(TICKER).name, "exists:", memory.cache_path(TICKER).exists())
n_before = sum(1 for _ in config.TRACE_PATH.open(encoding="utf-8"))
t0 = time.time()
state2 = run_pipeline(TICKER)          # use_cache=True (default)
n_after = sum(1 for _ in config.TRACE_PATH.open(encoding="utf-8"))
new_records = [json.loads(l) for l in config.TRACE_PATH.read_text(encoding="utf-8").splitlines()[n_before:]]
print(f"cache_hit={state2['cache_hit']} in {time.time() - t0:.2f} s; tool calls in second run: "
      f"{sum(r['event'] == 'tool_call' for r in new_records)}")
assert state2["cache_hit"] and state2["final_report"] == final

Cached file: MSFT_2026-10-08.json exists: True
[memory] CACHE HIT MSFT_2026-10-08.json -> loading brief, skipping all tools
cache_hit=True in 0.07 s; tool calls in second run: 0


## 3C.3 `agent_trace.jsonl`: one record per tool call (tool, inputs, output ≤ 200 chars, duration)

In [13]:
trace = pd.DataFrame([json.loads(l) for l in config.TRACE_PATH.read_text(encoding="utf-8").splitlines()])
calls = trace[trace.event == "tool_call"]
print(f"{len(trace)} records ({len(calls)} tool calls) in {config.TRACE_PATH.relative_to(Path.cwd())}")
print("max output length:", calls["output"].str.len().max(), "chars")
display(calls[["ts", "agent", "tool", "inputs", "status", "duration_ms", "output"]].tail(12))
display(calls.groupby(["agent", "tool"]).agg(calls=("tool", "size"), mean_ms=("duration_ms", "mean"),
                                            errors=("status", lambda s: int((s != "ok").sum()))).round(0))

26 records (21 tool calls) in logs\agent_trace.jsonl
max output length: 200 chars


,ts,agent,tool,inputs,status,duration_ms,output
9,2026-10-08T13:02:41.173+00:00,research_agent,get_news,"{'ticker': 'MSFT', 'n': 10}",error,0.0,"{""error"": ""get_news unavailable (simulated provider outage)"", ""hint"": ""use an alternative tool or retry with differe..."
10,2026-10-08T13:02:46.598+00:00,research_agent,web_search,"{'query': 'Microsoft Corp recent news October 2026', 'max_results': 5}",ok,4726.7,"[{""title"": ""Windows and Surface October 2026 news - news.microsoft.com"", ""snippet"": ""Join us for the latest on Windo..."
11,2026-10-08T13:02:47.625+00:00,research_agent,calculate_volatility,"{'ticker': 'MSFT', 'window': 30}",ok,503.8,"{""ticker"": ""MSFT"", ""window_days"": 30, ""annualised_vol_pct"": 22.01, ""one_year_vol_pct"": 32.3, ""vol_percentile_1y"": 32..."
12,2026-10-08T13:02:49.266+00:00,research_agent,llm_sentiment,"{'headlines': ['Windows and Surface October 2026 news - news.microsoft.com', 'Windows And Surface Event October 2026...",ok,712.7,"{""score"": 0.0, ""label"": ""neutral"", ""n_headlines"": 5, ""counts"": {""positive"": 0, ""neutral"": 5, ""negative"": 0}, ""most_n..."
13,2026-10-08T13:02:52.869+00:00,research_agent,web_search,"{'query': 'Microsoft analyst risk outlook Q4 2026 cloud competition AI regulation', 'max_results': 10}",ok,2812.9,"[{""title"": ""Microsoft Q4 FY 2026: AI Demand Accelerates Azure and ..."", ""snippet"": ""Jul 31, 2026 · Futurum Research ..."
14,2026-10-08T13:03:32.872+00:00,data_analyst,web_search,{'query': 'MSFT'},denied,0.0,"{""error"": ""PermissionError: tool 'web_search' is not available to data_analyst"", ""allowed_tools"": [""calculate_volati..."
15,2026-10-08T13:03:32.875+00:00,research_writer,get_price_data,{'ticker': 'MSFT'},denied,0.0,"{""error"": ""PermissionError: tool 'get_price_data' is not available to research_writer"", ""allowed_tools"": [""get_news""..."
16,2026-10-08T13:03:33.909+00:00,data_analyst,get_price_data,"{'ticker': 'MSFT', 'period': '1y'}",ok,326.8,"{""ticker"": ""MSFT"", ""period"": ""1y"", ""n_bars"": 252, ""start"": ""2025-10-07"", ""end"": ""2026-10-07"", ""last_close"": 529.76, ..."
17,2026-10-08T13:03:35.128+00:00,data_analyst,calculate_volatility,"{'ticker': 'MSFT', 'window': 30}",ok,556.4,"{""ticker"": ""MSFT"", ""window_days"": 30, ""annualised_vol_pct"": 22.01, ""one_year_vol_pct"": 32.3, ""vol_percentile_1y"": 32..."
19,2026-10-08T13:03:51.311+00:00,research_writer,get_news,"{'ticker': 'MSFT', 'n': 10}",ok,4172.4,"[{""headline"": ""Microsoft Xbox broadens entertainment push with new XP division (MSFT:NASDAQ)"", ""publisher"": ""Seeking..."


calls  mean_ms  errors
agent           tool                                        
data_analyst    calculate_volatility      1    556.0       0
                get_price_data            1    327.0       0
                llm_sentiment             1    859.0       0
                web_search                1      0.0       1
research_agent  calculate_volatility      1    504.0       0
                get_news                  1      0.0       1
                get_price_data            1    398.0       0
                llm_sentiment             1    713.0       0
                web_search                2   3770.0       0
research_writer get_news                  1   4172.0       0
                get_price_data            1      0.0       1
                web_search                1   2339.0       0
tool_smoke_test calculate_volatility      2    275.0       1
                get_news                  1   6241.0       0
                get_price_data            2   1876.0       1
                llm_sentiment             2   2567.0       1
                web_search                1   5108.0       0

---
## Bonus: observability dashboard
`streamlit run task3_agentic/dashboard/app.py` opens a viewer for `agent_trace.jsonl` with KPIs, latency per tool, a filterable tool-call timeline, and expandable handoff payloads. A screenshot is in the task README.